<a href="https://colab.research.google.com/github/ridmikaw/ML-Assignment-Logistic-Regression/blob/main/IT22088246.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 💳 Credit Card Fraud Detection using Logistic Regression
**IT Number - IT22088246**

**Assignment - Supervised Machine Learning**  
**Algorithm: Logistic Regression**  
**Dataset: Credit Card Fraud Detection (284,808 records)**

---
## 📚 Step 1: Import Libraries

In [1]:
# ── Data manipulation ────────────────────────────────────────
import numpy  as np
import pandas as pd

# ── Visualisation ────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn           as sns

# ── Preprocessing ────────────────────────────────────────────
from sklearn.preprocessing   import StandardScaler
from sklearn.model_selection  import train_test_split, StratifiedKFold, cross_val_score

# ── Handling class imbalance ─────────────────────────────────
from imblearn.over_sampling  import SMOTE
from collections             import Counter

# ── Logistic Regression model ────────────────────────────────
from sklearn.linear_model    import LogisticRegression

# ── Evaluation metrics ───────────────────────────────────────
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve,
    average_precision_score
)

# ── Install imbalanced-learn (for SMOTE) ─────────────────────
!pip install -q imbalanced-learn

# ── Reproducibility seed ─────────────────────────────────────
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ── Plot style ───────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 6)})

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


---
## 📦 Step 2: Dataset Description & Load from Google Drive

### About the Dataset
| Property | Detail |
|---|---|
| **Source** | [Kaggle – Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) |
| **Provider** | ULB Machine Learning Group, Belgium |
| **Records** | 284,808 transactions |
| **Features** | 30 (28 PCA components + Time + Amount) |
| **Target** | `Class` → 0 = Legitimate, 1 = Fraud |
| **Fraud ratio** | ~0.17 % (492 fraudulent out of 284,807) |
| **Time span** | 2 days of European card-holder transactions (Sept 2013) |
| **Storage** | Loaded directly from Google Drive |

**Feature notes:**
- **V1–V28**: Principal components obtained via PCA (original features are anonymised for confidentiality)
- **Time**: Seconds elapsed since the first transaction in the dataset
- **Amount**: Transaction amount in EUR
- **Class**: Binary target label (0 = Legitimate, 1 = Fraud)

In [3]:
from google.colab import drive

# ── 3.1  Mount Google Drive ──────────────────────────────────
drive.mount('/content/drive')
print("✅ Google Drive mounted at /content/drive")

Mounted at /content/drive
✅ Google Drive mounted at /content/drive


In [4]:
import os

# directly in My Drive root
DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/ml-assignment/creditcard.csv'

# ── Verify the file exists before loading ───────────────────
if os.path.exists(DATASET_PATH):
    file_size_mb = os.path.getsize(DATASET_PATH) / (1024 * 1024)
    print(f"✅ Found: {DATASET_PATH}")
    print(f"   File size: {file_size_mb:.1f} MB")
else:
    # If not found, search Drive automatically
    print("⚠️  File not found at default path. Searching Drive...")
    result = !find /content/drive/MyDrive -name 'creditcard.csv' 2>/dev/null
    if result:
        DATASET_PATH = result[0]
        print(f"✅ Auto-found at: {DATASET_PATH}")
        print(f"   Update DATASET_PATH above to this path for next run.")
    else:
        print("❌ creditcard.csv not found in Google Drive.")
        print("   Please upload it to your Drive and update DATASET_PATH.")

✅ Found: /content/drive/MyDrive/Colab Notebooks/ml-assignment/creditcard.csv
   File size: 143.8 MB


In [5]:
# ── 3.3  Load CSV into a Pandas DataFrame ───────────────────
print("Loading dataset from Google Drive...")
df = pd.read_csv(DATASET_PATH)

print(f"\n✅ Dataset loaded successfully!")
print(f"   Shape         : {df.shape}")
print(f"   Total records : {len(df):,}")
print(f"   Total columns : {df.shape[1]}")
print(f"\nColumn names:")
print(list(df.columns))
print("\nFirst 5 rows:")
df.head()

Loading dataset from Google Drive...

✅ Dataset loaded successfully!
   Shape         : (284807, 31)
   Total records : 284,807
   Total columns : 31

Column names:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']

First 5 rows:


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


---
## 🔍 Step 3: Exploratory Data Analysis (EDA)

In [6]:
# ── 4.1  Basic statistics ────────────────────────────────────
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Rows    : {df.shape[0]:,}")
print(f"Columns : {df.shape[1]}")
print(f"\nData types:")
print(df.dtypes.value_counts())
print(f"\nMissing values:\n{df.isnull().sum().sum()} total")
print("\nDescriptive statistics (Amount & Time):")
df[['Time','Amount']].describe()

DATASET OVERVIEW
Rows    : 284,807
Columns : 31

Data types:
float64    30
int64       1
Name: count, dtype: int64

Missing values:
0 total

Descriptive statistics (Amount & Time):


,Time,Amount
count,284807.000000,284807.000000
mean,94813.859575,88.349619
std,47488.145955,250.120109
min,0.000000,0.000000
25%,54201.500000,5.600000
50%,84692.000000,22.000000
75%,139320.500000,77.165000
max,172792.000000,25691.160000
